In [ ]:
!pip -q install transformers accelerate bitsandbytes sentence-transformers faiss-cpu stix2 requests fastapi uvicorn pyngrok
#cell1

In [ ]:
from huggingface_hub import login
login()
#cell2

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "fdtn-ai/Foundation-Sec-8B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("✅ Foundation-Sec loaded successfully")

#cell3


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

✅ Foundation-Sec loaded successfully


In [ ]:
pipeline_config = {
    "name": "AI-Driven SOAR IoC-to-Rule Pipeline",
    "version": "1.0",

    "stages": [
        {
            "name": "llm_mitre_mapping",
            "description": "LLM maps IOC + context to MITRE ATT&CK techniques"
        },
        {
            "name": "rag_validation",
            "description": "Validate mappings using ATT&CK knowledge + evidence profiles"
        },
        {
            "name": "rule_generation",
            "description": "Generate detection rules from validated techniques"
        },
        {
            "name": "deployment",
            "description": "Deploy rules into Wazuh SIEM"
        }
    ]
}

print("✅ Pipeline config loaded")


✅ Pipeline config loaded


In [ ]:
import requests, zipfile, io

MITRE_URL = "https://github.com/mitre/cti/archive/refs/heads/master.zip"

r = requests.get(MITRE_URL)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall("mitre_cti")

print("✅ MITRE ATT&CK downloaded")

#cell4


✅ MITRE ATT&CK downloaded


In [ ]:
from stix2 import MemoryStore, Filter
import json

store = MemoryStore()
store.load_from_file(
    "mitre_cti/cti-master/enterprise-attack/enterprise-attack.json"
)

attack_index = {}

attack_patterns = store.query([
    Filter("type", "=", "attack-pattern")
])

for obj in attack_patterns:
    # Extract ATT&CK technique ID (Txxxx)
    external_refs = obj.get("external_references", [])
    tid = None
    for ref in external_refs:
        if ref.get("source_name") == "mitre-attack":
            tid = ref.get("external_id")
            break

    if not tid:
        continue

    attack_index[tid] = {
        "name": obj.name,
        "tactics": [
            phase["phase_name"]
            for phase in obj.get("kill_chain_phases", [])
        ],
        "description": obj.description
    }

with open("attack_techniques_index.json", "w") as f:
    json.dump(attack_index, f, indent=2)

print(f"✅ Indexed {len(attack_index)} ATT&CK techniques")

#cell5


✅ Indexed 835 ATT&CK techniques


In [ ]:
evidence_profiles = {
    "ioc_type_to_evidence": {
        "process_command": [
            "process_creation",
            "command_line",
            "script_interpreter"
        ],
        "file_path": [
            "file_creation",
            "file_execution",
            "file_modification"
        ],
        "domain": [
            "dns_query",
            "network_connection"
        ],
        "ip": [
            "network_connection"
        ],
        "url": [
            "http_request",
            "network_connection"
        ],
        "registry_key": [
            "registry_modification",
            "registry_query"
        ],
        "file_hash": [
            "file_execution",
            "process_execution"
        ],
        "email": [
            "email_delivery",
            "network_connection"
        ],
    }
}

# ------------------------------------------------------------------
# NEW: Which IOC types can produce evidence for each technique.
#
# Design rule:
#   - Only include an IOC type if a SOC analyst could realistically
#     detect that technique FROM that IOC type alone.
#   - Example: T1003.001 (LSASS dump) requires process_command or
#     file_hash — you cannot observe credential dumping from a
#     domain or IP IOC alone.
#   - "any" = technique is observable from all IOC types (rare).
#
# If a technique is NOT listed here, it passes through unchecked
# (safe fallback — we never block unknown techniques).
# ------------------------------------------------------------------
TECHNIQUE_IOC_ALLOWLIST = {

    # ── EXECUTION ──────────────────────────────────────────────────
    "T1059.001": {"process_command", "file_hash"},          # PowerShell
    "T1059.003": {"process_command", "file_hash"},          # CMD
    "T1059.005": {"process_command", "file_hash"},          # VBScript
    "T1059.007": {"process_command", "file_hash", "url"},   # JavaScript
    "T1047":     {"process_command", "file_hash"},          # WMI
    "T1053.005": {"process_command", "registry_key"},       # Schtasks
    "T1218.005": {"process_command", "url", "domain"},      # Mshta
    "T1218.010": {"process_command", "file_hash"},          # Regsvr32
    "T1218.011": {"process_command", "file_hash"},          # Rundll32

    # ── PERSISTENCE ────────────────────────────────────────────────
    "T1547.001": {"registry_key", "process_command"},       # Registry Run Keys
    "T1547.004": {"registry_key"},                          # Winlogon
    "T1543.003": {"registry_key", "process_command"},       # New service
    "T1546.012": {"registry_key"},                          # IFEO
    "T1053.002": {"process_command"},                       # At.exe

    # ── DEFENSE EVASION ────────────────────────────────────────────
    "T1027":     {"process_command", "file_hash"},          # Obfuscation
    "T1036":     {"process_command", "file_hash"},          # Masquerading
    "T1055":     {"process_command", "file_hash"},          # Process injection
    "T1562.001": {"process_command", "registry_key"},       # Disable defenses
    "T1070.004": {"process_command", "file_hash"},          # File deletion
    "T1490":     {"process_command"},                       # Shadow copy delete

    # ── CREDENTIAL ACCESS ──────────────────────────────────────────
    "T1003.001": {"process_command", "file_hash"},          # LSASS dump
    "T1003.002": {"process_command", "file_hash"},          # SAM/SYSTEM hive
    "T1110":     {"ip", "domain"},                          # Brute force
    "T1555":     {"process_command", "file_hash"},          # Credentials from store
    "T1056.001": {"process_command", "file_hash"},          # Keylogging

    # ── DISCOVERY ──────────────────────────────────────────────────
    "T1082":     {"process_command"},                       # System info
    "T1046":     {"process_command", "ip"},                 # Network scan
    "T1087.001": {"process_command"},                       # Local account enum
    "T1087.002": {"process_command", "domain"},             # Domain account enum
    "T1083":     {"process_command"},                       # File enumeration
    "T1135":     {"process_command", "ip"},                 # Network share enum
    "T1057":     {"process_command"},                       # Process discovery

    # ── LATERAL MOVEMENT ───────────────────────────────────────────
    "T1021.001": {"ip"},                                    # RDP
    "T1021.002": {"ip"},                                    # SMB/admin shares
    "T1021.006": {"ip", "process_command"},                 # WinRM
    "T1550.002": {"ip", "process_command"},                 # Pass the hash
    "T1570":     {"ip", "file_hash"},                       # Lateral tool transfer

    # ── COMMAND AND CONTROL ────────────────────────────────────────
    "T1071.001": {"ip", "domain", "url"},                   # Web protocols C2
    "T1071.004": {"domain", "ip"},                          # DNS C2
    "T1090":     {"ip", "domain", "url"},                   # Proxy
    "T1090.003": {"ip", "domain"},                          # Multi-hop proxy (Tor)
    "T1095":     {"ip"},                                    # Non-app layer protocol
    "T1571":     {"ip"},                                    # Non-standard port
    "T1573":     {"ip", "domain"},                          # Encrypted channel
    "T1105":     {"ip", "domain", "url", "file_hash"},      # Ingress tool transfer

    # ── EXFILTRATION ───────────────────────────────────────────────
    "T1041":     {"ip", "domain"},                          # Exfil over C2
    "T1567":     {"ip", "domain", "url"},                   # Exfil over web service
    "T1048":     {"ip", "domain"},                          # Exfil over alt protocol

    # ── INITIAL ACCESS ─────────────────────────────────────────────
    "T1566.001": {"email", "file_hash"},                    # Spearphishing attachment
    "T1566.002": {"email", "url"},                          # Spearphishing link
    "T1190":     {"ip", "url"},                             # Exploit public-facing app
    "T1133":     {"ip", "domain"},                          # External remote services

    # ── IMPACT ─────────────────────────────────────────────────────
    "T1486":     {"file_hash", "process_command"},          # Data encrypted (ransomware)
    "T1489":     {"process_command"},                       # Service stop
    "T1491":     {"ip", "domain", "url"},                   # Defacement
}

print(f"✅ Evidence profiles loaded")
print(f"✅ TECHNIQUE_IOC_ALLOWLIST loaded: {len(TECHNIQUE_IOC_ALLOWLIST)} techniques with IOC restrictions")

✅ Evidence profiles loaded
✅ TECHNIQUE_IOC_ALLOWLIST loaded: 55 techniques with IOC restrictions


In [ ]:
import json

with open("attack_techniques_index.json", "w") as f:
    json.dump(attack_index, f, indent=2)

print("✅ attack_techniques_index.json saved")

#cell6


✅ attack_techniques_index.json saved


In [ ]:
def build_chat_prompt(ioc_type: str, ioc: str, context: str):
    system_message = """
You are a senior SOC / DFIR analyst specializing in MITRE ATT&CK mapping.

Your task is to map the given Indicator of Compromise (IOC) to MITRE ATT&CK
techniques based STRICTLY on observable behavior described in the context.

Rules:
- Only propose techniques that are directly supported by the context
- Do NOT guess, infer, or hallucinate techniques not evidenced by the context
- Each technique MUST have a specific behavioral justification (not generic)
- Technique IDs must be real ATT&CK IDs in format Txxxx or Txxxx.xxx
- Output MUST be valid JSON only — no explanation, no markdown, no preamble
- If the context does not support any technique, return an empty techniques list
""".strip()

    user_message = f"""IOC Type: {ioc_type}
IOC: {ioc}
Context: {context}

Respond ONLY with valid JSON in this exact format:
{{
  "ioc_type": "{ioc_type}",
  "ioc": "{ioc}",
  "techniques": [
    {{
      "id": "Txxxx.xxx",
      "reason": "Specific behavioral justification tied to the context above"
    }}
  ]
}}"""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return prompt

#cell8

In [ ]:
def build_apt_projection_prompt(
    ioc_type: str,
    ioc: str,
    context: str,
    detected_techniques: list,
    candidate_apts: list,
    candidate_next_techniques: list,
    enrichment_summary: dict = None
):
    """
    Strict APT progression projection prompt.
    Forces model to choose ONLY from provided candidate_next_techniques.
    """

    system_message = """
You are a senior Threat Intelligence analyst.

Your task is to determine the MOST LIKELY next MITRE ATT&CK technique
based strictly on:

- Already detected techniques
- Selected candidate APT group
- Known historical techniques used by that APT
- Observed behavior context
- Optional enrichment intelligence

CRITICAL RULES:

- If you output a technique not present in CANDIDATE_NEXT_TECHNIQUES,
your answer will be discarded.
- You MUST copy the technique ID EXACTLY as written in the list.
- No variations.
- No substitutions.
- No new techniques.
- You MUST NOT output any text outside valid JSON.
""".strip()

    user_message = f"""
IOC TYPE:
{ioc_type}

IOC VALUE:
{ioc}

CONTEXT:
{context}

DETECTED TECHNIQUES:
{detected_techniques}

CANDIDATE APT GROUPS (ranked by relevance):
{candidate_apts}

CANDIDATE_NEXT_TECHNIQUES (YOU MUST CHOOSE FROM THIS LIST ONLY):
{candidate_next_techniques}

OPTIONAL ENRICHMENT:
{enrichment_summary if enrichment_summary else "None"}

Select the SINGLE most probable next technique.

Respond ONLY with valid JSON:

{{
  "predicted_next_technique": {{
    "id": "Txxxx[.xxx]",
    "name": "Technique Name",
    "tactic": "MITRE Tactic",
    "why_this_is_next": "Clear logical progression explanation"
  }},
  "confidence": {{
    "score": 0-100,
    "level": "High | Medium | Low",
    "justification": "Why this confidence level was assigned"
  }}
}}
""".strip()

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return prompt


In [ ]:
import json, re

TECH_ID_RE = re.compile(r"^T\d{4}(\.\d{3})?$", re.IGNORECASE)

def _extract_all_json_objects(text: str):
    """
    Returns ALL top-level JSON objects found via brace balancing.
    """
    # Remove model tokens like <|assistant|> etc
    text = re.sub(r"<\|.*?\|>", "", text)

    objs = []
    start_positions = [m.start() for m in re.finditer(r"\{", text)]
    for start in start_positions:
        brace = 0
        for i in range(start, len(text)):
            if text[i] == "{":
                brace += 1
            elif text[i] == "}":
                brace -= 1
                if brace == 0:
                    candidate = text[start:i+1]
                    try:
                        objs.append(json.loads(candidate))
                    except Exception:
                        pass
                    break
    return objs

def _score_mapping_obj(obj: dict) -> int:
    """
    Higher score = more likely it's the REAL answer (not the template).
    """
    if not isinstance(obj, dict):
        return -999

    score = 0
    if "ioc_type" in obj: score += 2
    if "ioc" in obj: score += 2
    if "techniques" in obj and isinstance(obj["techniques"], list):
        score += 3
        techs = obj["techniques"]

        # count real technique IDs
        real = 0
        placeholders = 0
        for t in techs:
            if isinstance(t, dict):
                tid = str(t.get("id", "")).strip()
                if TECH_ID_RE.match(tid):
                    real += 1
                if "Txxxx" in tid or "xxxx" in tid.lower():
                    placeholders += 1

        score += real * 5         # strongly prefer real IDs
        score -= placeholders * 8 # strongly reject template

        # small bonus if there is a reason
        for t in techs:
            if isinstance(t, dict) and t.get("reason"):
                score += 1

    return score

def extract_json_from_text(text: str):
    """
    Extract the BEST matching JSON object:
    - Not the template
    - Contains real ATT&CK technique IDs
    """
    objs = _extract_all_json_objects(text)
    if not objs:
        print("❌ JSON EXTRACTION FAILED: No JSON objects found")
        return None

    scored = sorted([(o, _score_mapping_obj(o)) for o in objs], key=lambda x: x[1], reverse=True)
    best_obj, best_score = scored[0]

    # Debug: if you want to see what it chose
    # print("Top JSON scores:", [s for _, s in scored[:3]])

    if best_score < 3:
        print("❌ JSON EXTRACTION FAILED: Only weak/template-like JSON found")
        return None

    return best_obj


In [ ]:
llm_mapping_config = {
    "model": "Foundation-Sec-8B-Instruct",
    "temperature": 0.0,
    "max_tokens": 256,

    "policy": {
        "allow_guessing": False,
        "require_context": True,
        "multi_technique_allowed": True
    },

    "output_schema": {
        "required_fields": ["ioc_type", "ioc", "techniques"]
    }
}

print("✅ LLM mapping policy loaded")


✅ LLM mapping policy loaded


In [ ]:
def llm_only_map(ioc_type: str, ioc: str, context: str, max_new_tokens: int = 200):
    """
    LLM-only MITRE mapping (NO RAG).
    Used to observe raw model behavior.
    """

    prompt = build_chat_prompt(ioc_type, ioc, context)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    print("\n================ RAW MODEL OUTPUT ================\n")
    print(decoded)
    print("\n=================================================\n")

    parsed = extract_json_from_text(decoded)

    if parsed is None:
        print("❌ JSON EXTRACTION FAILED")
        return None

    print("✅ JSON EXTRACTION SUCCESSFUL")
    return parsed


In [ ]:
result = llm_only_map(
    ioc_type="process_command",
    ioc="powershell.exe -enc SQBFAFgA...",
    context="Encoded PowerShell execution observed spawning from cmd.exe"
)

print(json.dumps(result, indent=2))


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



================ RAW MODEL OUTPUT ================

<|begin_of_text|><|begin_of_text|><|system|>
You are a senior SOC / DFIR analyst specializing in MITRE ATT&CK mapping.

Your task is to map the given Indicator of Compromise (IOC) to MITRE ATT&CK
techniques based STRICTLY on observable behavior described in the context.

Rules:
- Only propose techniques that are directly supported by the context
- Do NOT guess, infer, or hallucinate techniques not evidenced by the context
- Each technique MUST have a specific behavioral justification (not generic)
- Technique IDs must be real ATT&CK IDs in format Txxxx or Txxxx.xxx
- Output MUST be valid JSON only — no explanation, no markdown, no preamble
- If the context does not support any technique, return an empty techniques list
<|user|>
IOC Type: process_command
IOC: powershell.exe -enc SQBFAFgA...
Context: Encoded PowerShell execution observed spawning from cmd.exe

Respond ONLY with valid JSON in this exact format:
{
  "ioc_type": "process_

In [ ]:
import json

ATTACK_INDEX_PATH = "attack_techniques_index.json"

# Load full MITRE ATT&CK metadata (dict keyed by technique ID)
with open(ATTACK_INDEX_PATH, "r") as f:
    ATTACK_META_INDEX = json.load(f)

# Build a separate ID-only set for RAG validation
ATTACK_ID_SET = set(ATTACK_META_INDEX.keys())

print(f"✅ Loaded {len(ATTACK_ID_SET)} ATT&CK techniques")
print("✅ Full MITRE metadata preserved (name, tactics, description)")


✅ Loaded 835 ATT&CK techniques
✅ Full MITRE metadata preserved (name, tactics, description)


In [ ]:
rag_validation_config = {
    "knowledge_sources": [
        "MITRE_ATT&CK",
        "Evidence_Profiles"
    ],

    "rules": [
        "technique_must_exist",
        "ioc_type_must_have_evidence",
        "no_evidence_no_mapping"
    ],

    "validated_tag": "RAG_ATT&CK"
}

print("✅ RAG validation config loaded")


✅ RAG validation config loaded


In [ ]:
def rag_validate_techniques(
    llm_output: dict,
    attack_index: set,
    evidence_profiles: dict
):
    """
    Deep RAG validation layer — three rules:

    Rule 1 — Technique must exist in MITRE ATT&CK index.
    Rule 2 — IOC type must have at least one evidence category
              (same as before — broad gate).
    Rule 3 — NEW: If the technique is in TECHNIQUE_IOC_ALLOWLIST,
              the IOC type must be in its allowed set.
              Example: T1003.001 (LSASS dump) is blocked for domain/ip/url/email.
              Techniques NOT in the allowlist pass through (safe fallback).

    Each rejected technique is recorded with a reason for transparency.
    """

    if llm_output is None:
        raise ValueError("LLM output is None — cannot validate")

    ioc_type   = llm_output.get("ioc_type")
    ioc_value  = llm_output.get("ioc")
    techniques = llm_output.get("techniques", [])

    if not ioc_type or not techniques:
        return {
            "ioc_type":  ioc_type,
            "ioc":       ioc_value,
            "techniques": [],
            "rejected":  []
        }

    # Normalise ioc_type to match allowlist keys
    # (notebook uses "hash"/"registry" in some places)
    ioc_type_norm = ioc_type.strip().lower()
    if ioc_type_norm in {"hash", "filehash"}:
        ioc_type_norm = "file_hash"
    if ioc_type_norm in {"registry", "regkey"}:
        ioc_type_norm = "registry_key"

    allowed_evidence = evidence_profiles.get(ioc_type_norm, [])

    validated = []
    rejected  = []   # ← NEW: transparency log

    for t in techniques:
        tid    = (t.get("id") or "").strip().upper()
        reason = t.get("reason", "")

        # ── Rule 1: technique must exist in ATT&CK ──────────────────
        if tid not in attack_index:
            rejected.append({
                "id":     tid,
                "reason": "Not found in MITRE ATT&CK index"
            })
            continue

        # ── Rule 2: IOC type must have evidence categories ───────────
        if not allowed_evidence:
            rejected.append({
                "id":     tid,
                "reason": f"No evidence categories defined for IOC type '{ioc_type_norm}'"
            })
            continue

        # ── Rule 3 (NEW): IOC-technique compatibility check ──────────
        allowed_ioc_types = TECHNIQUE_IOC_ALLOWLIST.get(tid)

        if allowed_ioc_types is not None:
            # Technique IS in the allowlist — enforce the restriction
            if ioc_type_norm not in allowed_ioc_types:
                rejected.append({
                    "id":     tid,
                    "reason": (
                        f"IOC type '{ioc_type_norm}' cannot produce observable evidence "
                        f"for {tid}. Allowed IOC types: {sorted(allowed_ioc_types)}"
                    )
                })
                continue
        # else: technique not in allowlist → pass through (safe fallback)

        validated.append({
            "id":           tid,
            "reason":       reason,
            "validated_by": "RAG_ATT&CK_v2"   # bumped tag so you can tell which version ran
        })

    # Print rejection summary (useful during development)
    if rejected:
        print(f"⚠️  RAG rejected {len(rejected)} technique(s):")
        for r in rejected:
            print(f"   ✗ {r['id']}: {r['reason']}")

    print(f"✅ RAG validated {len(validated)}/{len(techniques)} techniques for IOC type '{ioc_type_norm}'")

    return {
        "ioc_type":   ioc_type,
        "ioc":        ioc_value,
        "techniques": validated,
        "rejected":   rejected   # ← NEW: included in output for transparency
    }

In [ ]:
validated = rag_validate_techniques(
    llm_output=result,
    attack_index=ATTACK_ID_SET,
    evidence_profiles=evidence_profiles["ioc_type_to_evidence"]
)

print(json.dumps(validated, indent=2))


✅ RAG validated 1/1 techniques for IOC type 'process_command'
{
  "ioc_type": "process_command",
  "ioc": "powershell.exe -enc SQBFAFgA...",
  "techniques": [
    {
      "id": "T1027.006",
      "reason": "Encoded command line arguments used to evade detection and execute malicious PowerShell commands.",
      "validated_by": "RAG_ATT&CK_v2"
    }
  ],
  "rejected": []
}


In [ ]:
# Clone Sigma official rules repository
!git clone https://github.com/SigmaHQ/sigma.git


Cloning into 'sigma'...
remote: Enumerating objects: 128689, done.
remote: Counting objects: 100% (249/249), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 128689 (delta 159), reused 68 (delta 68), pack-reused 128440 (from 4)
Receiving objects: 100% (128689/128689), 45.93 MiB | 39.03 MiB/s, done.
Resolving deltas: 100% (98616/98616), done.


In [ ]:
import os

SIGMA_BASE_DIR = "sigma/rules"

print("Sigma rules directory exists:", os.path.exists(SIGMA_BASE_DIR))


Sigma rules directory exists: True


In [ ]:
import yaml
import os

def load_sigma_rules(base_dir: str):
    rules = []

    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith((".yml", ".yaml")):
                path = os.path.join(root, file)
                try:
                    with open(path, "r", encoding="utf-8") as f:
                        rule = yaml.safe_load(f)
                        if rule:
                            rule["_file_path"] = path
                            rules.append(rule)
                except Exception:
                    continue

    return rules

In [ ]:
def normalize_sigma_rule(rule: dict):
    return {
        "title": rule.get("title"),
        "id": rule.get("id"),
        "description": rule.get("description"),
        "status": rule.get("status"),
        "level": rule.get("level"),
        "tags": rule.get("tags", []),
        "logsource": rule.get("logsource", {}),
        "detection": rule.get("detection", {}),
        "file_path": rule.get("_file_path")
    }


In [ ]:
IOC_TO_LOGSOURCE = {
    "process_command": {
        "category": "process_creation",
        "product": "windows"
    },
    "file_path": {
        "category": "file_event",
        "product": "windows"
    },
    "domain": {
        "category": "dns_query",
        "product": "windows"
    },
    "ip": {
        "category": "network_connection",
        "product": "windows"
    },
    "url": {
        "category": "proxy",
        "product": "windows"
    },
    "registry": {
        "category": "registry_event",
        "product": "windows"
    }
}


In [ ]:
def filter_sigma_by_mitre(rules: list, validated_techniques: list):
    valid_ids = {t.lower() for t in validated_techniques}
    matched = []

    for rule in rules:
        tags = rule.get("tags", [])
        for tid in valid_ids:
            if any(tid in tag.lower() for tag in tags):
                matched.append(rule)
                break

    return matched


In [ ]:
def filter_by_ioc_type(rules, ioc_type):
    expected = IOC_TO_LOGSOURCE.get(ioc_type)
    if not expected:
        return []

    filtered = []

    for rule in rules:
        logsource = rule.get("logsource", {})
        if (
            logsource.get("category") == expected["category"]
            and logsource.get("product") == expected["product"]
        ):
            filtered.append(rule)

    return filtered


In [ ]:
# =========================================
# 🔥 IOC-TYPE STRUCTURAL SCORING ENGINE
# =========================================

IOC_FIELD_RELEVANCE = {
    "ip": {
        "DestinationIp", "SourceIp"
    },
    "domain": {
        "QueryName", "DestinationHostname"
    },
    "url": {
        "Url", "RequestUrl", "TargetUrl"
    },
    "file_hash": {
        "Hashes", "Imphash", "Sha256", "Md5"
    },
    "email": {
        "Sender", "Recipient", "Subject"
    },
    "process_command": {
        "CommandLine", "Image", "ParentImage"
    },
    "registry_key": {
        "TargetObject"
    }
}


def extract_sigma_fields(rule):
    """
    Extract base detection field names from Sigma rule.
    """
    detection = rule.get("detection", {})
    fields = set()

    for key, value in detection.items():
        if key == "condition":
            continue

        if isinstance(value, dict):
            for field in value.keys():
                base = field.split("|")[0]
                fields.add(base)

        elif isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    for field in item.keys():
                        base = field.split("|")[0]
                        fields.add(base)

    return fields


def score_sigma_rule(rule, ioc_type):
    """
    Score rule based purely on structural IOC relevance.
    """
    expected_fields = IOC_FIELD_RELEVANCE.get(ioc_type, set())
    rule_fields = extract_sigma_fields(rule)

    overlap = expected_fields.intersection(rule_fields)

    return len(overlap)

In [ ]:
def specialize_sigma_rule(rule: dict, ioc_value: str):
    detection = rule.get("detection", {}).copy()

    for key, value in detection.items():
        if isinstance(value, dict):
            for k in value:
                value[k] = ioc_value

    rule["detection"] = detection
    rule["status"] = "experimental"
    rule["description"] = (rule.get("description") or "") + " (Auto-specialized by AI pipeline)"

    return rule


In [ ]:
SIGMA_TO_WAZUH_FIELD_MAP = {
    "Image": "win.eventdata.Image",
    "OriginalFileName": "win.eventdata.OriginalFileName",
    "ParentImage": "win.eventdata.ParentImage",
    "CommandLine": "win.eventdata.CommandLine",

    "TargetFilename": "win.eventdata.TargetFilename",
    "FileName": "win.eventdata.FileName",

    "TargetObject": "win.eventdata.TargetObject",
    "Details": "win.eventdata.Details",

    "DestinationIp": "win.eventdata.DestinationIp",
    "DestinationPort": "win.eventdata.DestinationPort",
    "SourceIp": "win.eventdata.SourceIp",

    "User": "win.eventdata.User",
    "LogonId": "win.eventdata.LogonId",

    "ScriptBlockText": "win.eventdata.ScriptBlockText",
}

In [ ]:
def normalize_sigma_selections(detection: dict):
    selections = []

    for key, value in detection.items():
        if key == "condition":
            continue

        # Case 1: selection is a list (OR logic)
        if isinstance(value, list):
            for item in value:
                selections.append(item)

        # Case 2: selection is a dict
        elif isinstance(value, dict):
            selections.append(value)

    return selections


In [ ]:
def sigma_detection_to_wazuh_conditions(detection: dict):
    conditions = []

    selections = normalize_sigma_selections(detection)

    for sel in selections:
        for field, value in sel.items():
            if "|" in field:
                base_field, modifier = field.split("|", 1)
            else:
                base_field, modifier = field, "equals"

            wazuh_field = SIGMA_TO_WAZUH_FIELD_MAP.get(base_field)
            if not wazuh_field:
                continue  # unsupported field

            values = value if isinstance(value, list) else [value]

            for v in values:
                conditions.append((wazuh_field, modifier, v))

    return conditions


In [ ]:
def summarize_sigma_rule(rule):
    return {
        "title": rule.get("title"),
        "id": rule.get("id"),
        "level": rule.get("level"),
        "tags": rule.get("tags", []),
        "logsource": rule.get("logsource", {}),
        "detection_keys": list(rule.get("detection", {}).keys())
    }


In [ ]:
def build_mitre_block(mitre_ids):
    if not mitre_ids:
        return ""
    lines = ["<mitre>"]
    for mid in mitre_ids:
        lines.append(f"  <id>{mid}</id>")
    lines.append("</mitre>")
    return "\n".join(lines)

In [ ]:
import re

def escape_regex(s: str) -> str:
    return re.escape(str(s))

def build_field_xml(field, modifier, value):

    if not field:
        return None

    v = str(value)

    if modifier in ("contains", "endswith", "startswith"):
        vv = escape_regex(v)

        if modifier == "contains":
            pattern = f".*{vv}.*"
        elif modifier == "endswith":
            pattern = f"{vv}$"
        else:  # startswith
            pattern = f"^{vv}"

        return f'<field name="{field}" type="pcre2">(?i){pattern}</field>'

    # default equals
    return f'<field name="{field}">{v}</field>'

In [ ]:
LOGSOURCE_TO_PARENT_SID = {
    ("windows", "process_creation"): 61600,      # Sysmon Event ID 1
    ("windows", "network_connection"): 61603,    # Sysmon Event ID 3
    ("windows", "dns_query"): 61500,             # DNS baseline
    ("windows", "registry_event"): 61613,        # Sysmon registry
    ("windows", "file_event"): 61602,            # Sysmon file
    ("windows", "proxy"): 62000                  # Proxy logs
}

def resolve_parent_sid(rule):
    logsource = rule.get("logsource", {})
    key = (logsource.get("product"), logsource.get("category"))
    return LOGSOURCE_TO_PARENT_SID.get(key)

In [ ]:
# =========================================
# 🔥 GENERIC IOC VALUE COMPATIBILITY CHECK
# =========================================

def rule_allows_ioc(rule, ioc_type, ioc_value):
    """
    Ensure that if a Sigma rule restricts a field relevant to the IOC,
    the IOC value is compatible with that restriction.
    """

    detection = rule.get("detection", {})
    ioc_value_str = str(ioc_value).lower()

    relevant_fields = IOC_FIELD_RELEVANCE.get(ioc_type, set())

    for key, value in detection.items():
        if key == "condition":
            continue

        if isinstance(value, dict):
            for field, field_value in value.items():

                base_field = field.split("|")[0]

                if base_field not in relevant_fields:
                    continue

                values = field_value if isinstance(field_value, list) else [field_value]

                # If rule restricts the field and IOC does NOT match any restriction → reject
                if values:
                    match_found = False

                    for v in values:
                        v_str = str(v).lower()

                        # Exact match
                        if ioc_value_str == v_str:
                            match_found = True
                            break

                        # Partial match (contains)
                        if ioc_value_str in v_str or v_str in ioc_value_str:
                            match_found = True
                            break

                    if not match_found:
                        return False

    return True

In [ ]:
def sigma_rule_to_wazuh_xml(rule, base_id=100000):
    """
    Convert ONE Sigma rule (dict) into a production-ready Wazuh XML rule.
    """

    # Resolve parent SID from logsource
    parent_sid = resolve_parent_sid(rule)
    if not parent_sid:
        return None  # Not deployable if no parent decoder SID

    # Severity mapping
    level_map = {
        "low": 5,
        "medium": 10,
        "high": 12,
        "critical": 15
    }
    level = level_map.get(rule.get("level"), 10)

    description = rule.get("title", "Sigma-derived detection")

    # Extract MITRE technique IDs
    mitre_ids = [
        tag.replace("attack.", "").upper()
        for tag in rule.get("tags", [])
        if isinstance(tag, str) and tag.startswith("attack.t")
    ]

    mitre_xml = build_mitre_block(mitre_ids)

    # Extract detection conditions
    conditions = sigma_detection_to_wazuh_conditions(rule.get("detection", {}))
    if not conditions:
        return None

    # Build <field> XML blocks
    field_blocks = []
    for field, modifier, value in conditions:
        field_xml = build_field_xml(field, modifier, value)
        if field_xml:
            field_blocks.append(field_xml)

    if not field_blocks:
        return None

    fields_xml = "\n    ".join(field_blocks)

    xml = f"""
<group name="windows,attack">
  <rule id="{base_id}" level="{level}">
    <if_sid>{parent_sid}</if_sid>
    {fields_xml}
    <description>{description}</description>
    {mitre_xml}
  </rule>
</group>
""".strip()

    return xml

In [ ]:
# =========================================
# ✅ CANONICAL IOC TYPES (must match your UI)
# =========================================
CANON_IOC_TYPES = {
    "ip", "domain", "url", "file_hash", "email", "process_command", "registry_key"
}

# =========================================
# ✅ WAZUH TELEMETRY PARENT SID MAP (yours)
# =========================================
# Use the same one you already have; keeping here for clarity
LOGSOURCE_TO_PARENT_SID = {
    ("windows", "process_creation"): 61600,      # Sysmon Event ID 1
    ("windows", "network_connection"): 61603,    # Sysmon Event ID 3
    ("windows", "dns_query"): 61500,             # DNS baseline (adjust if needed)
    ("windows", "registry_event"): 61613,        # Sysmon registry
    ("windows", "file_event"): 61602,            # Sysmon file
    ("windows", "proxy"): 62000                  # Proxy logs (adjust if needed)
}

# =========================================
# ✅ IOC → DEFAULT TELEMETRY "BASE RULE"
# (these guarantee coverage for every IOC)
# =========================================
IOC_BASE_BLUEPRINT = {
    "ip": {
        "logsource": ("windows", "network_connection"),
        "conditions": [
            # exact IP match
            ("win.eventdata.destinationIp", "equals", "$IOC")
        ]
    },
    "domain": {
        "logsource": ("windows", "dns_query"),
        "conditions": [
            ("win.eventdata.queryName", "equals", "$IOC")
        ]
    },
    "url": {
        "logsource": ("windows", "proxy"),
        "conditions": [
            ("win.eventdata.url", "contains", "$IOC")
        ]
    },
    "file_hash": {
        "logsource": ("windows", "file_event"),
        "conditions": [
            # You may need to adjust this based on your decoder fields
            ("win.eventdata.hashes", "contains", "$IOC")
        ]
    },
    "email": {
        # Only works if you ingest email gateway logs into Wazuh.
        # If you don't, it will be filtered out later due to missing parent SID.
        "logsource": ("windows", "proxy"),
        "conditions": [
            ("win.eventdata.sender", "contains", "$IOC")
        ]
    },
    "process_command": {
        "logsource": ("windows", "process_creation"),
        "conditions": [
            ("win.eventdata.commandLine", "contains", "$IOC")
        ]
    },
    "registry_key": {
        "logsource": ("windows", "registry_event"),
        "conditions": [
            ("win.eventdata.targetObject", "contains", "$IOC")
        ]
    }
}

# =========================================
# ✅ ALLOWLIST: fields permitted in generated rules
# (prevents hallucinated/unsupported fields)
# =========================================
ALLOWED_WAZUH_FIELDS = {
    "win.eventdata.destinationIp",
    "win.eventdata.sourceIp",
    "win.eventdata.destinationPort",
    "win.eventdata.queryName",
    "win.eventdata.url",
    "win.eventdata.hashes",
    "win.eventdata.commandLine",
    "win.eventdata.image",
    "win.eventdata.parentImage",
    "win.eventdata.targetObject",
    "win.eventdata.details",
    "win.eventdata.user",
    "win.eventdata.targetFilename",
}

ALLOWED_OPS = {"equals", "contains", "startswith", "endswith"}  # rendered safely by build_field_xml

In [ ]:
TECHNIQUE_BLUEPRINTS = {

    # =========================================
    # INITIAL ACCESS
    # =========================================
    "T1566.001": [
        {"name": "Phishing attachment via Office macro",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.parentImage", "contains", "WINWORD"),
                        ("win.eventdata.image", "endswith", "cmd.exe")]},
        {"name": "Phishing - PowerShell spawned from Office",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.parentImage", "contains", "WINWORD"),
                        ("win.eventdata.image", "endswith", "powershell.exe")]},
    ],
    "T1566.002": [
        {"name": "Phishing link - browser spawning script",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.parentImage", "contains", "chrome"),
                        ("win.eventdata.image", "endswith", "powershell.exe")]},
    ],
    "T1190": [
        {"name": "Exploit public app - web shell spawn",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.parentImage", "contains", "w3wp"),
                        ("win.eventdata.image", "endswith", "cmd.exe")]},
    ],
    "T1133": [
        {"name": "External remote service - VPN/RDP inbound",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "3389")]},
    ],
    "T1078": [
        {"name": "Valid account logon anomaly",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "net.exe"),
                        ("win.eventdata.commandLine", "contains", "use")]},
    ],
    "T1204.001": [
        {"name": "User execution - malicious link via browser",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.parentImage", "contains", "chrome"),
                        ("win.eventdata.image", "endswith", "mshta.exe")]},
    ],
    "T1204.002": [
        {"name": "User execution - malicious Office macro",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.parentImage", "contains", "EXCEL"),
                        ("win.eventdata.image", "endswith", "cmd.exe")]},
    ],

    # =========================================
    # EXECUTION
    # =========================================
    "T1059.001": [
        {"name": "PowerShell execution (generic)",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe")]},
        {"name": "PowerShell encoded command",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "-enc")]},
        {"name": "PowerShell download cradle",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "DownloadString")]},
        {"name": "PowerShell IEX in-memory",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "IEX")]},
    ],
    "T1059.003": [
        {"name": "CMD suspicious execution",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "cmd.exe"),
                        ("win.eventdata.commandLine", "contains", "/c")]},
        {"name": "CMD spawned from Office",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.parentImage", "contains", "WINWORD"),
                        ("win.eventdata.image", "endswith", "cmd.exe")]},
    ],
    "T1059.005": [
        {"name": "VBScript execution via cscript",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "cscript.exe"),
                        ("win.eventdata.commandLine", "contains", ".vbs")]},
        {"name": "VBScript execution via wscript",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "wscript.exe")]},
    ],
    "T1059.006": [
        {"name": "Python script execution",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "contains", "python"),
                        ("win.eventdata.commandLine", "contains", ".py")]},
    ],
    "T1059.007": [
        {"name": "JavaScript via wscript/cscript",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "wscript.exe"),
                        ("win.eventdata.commandLine", "contains", ".js")]},
    ],
    "T1047": [
        {"name": "WMI process creation",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "wmic.exe"),
                        ("win.eventdata.commandLine", "contains", "process call create")]},
        {"name": "WMI remote execution",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "wmic.exe"),
                        ("win.eventdata.commandLine", "contains", "/node:")]},
    ],
    "T1053.005": [
        {"name": "Scheduled task creation",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "schtasks.exe"),
                        ("win.eventdata.commandLine", "contains", "/create")]},
        {"name": "Scheduled task remote",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "schtasks.exe"),
                        ("win.eventdata.commandLine", "contains", "/s ")]},
    ],
    "T1053.002": [
        {"name": "At.exe scheduled task",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "at.exe")]},
    ],
    "T1218.005": [
        {"name": "Mshta remote script",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "mshta.exe"),
                        ("win.eventdata.commandLine", "contains", "http")]},
        {"name": "Mshta vbscript inline",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "mshta.exe"),
                        ("win.eventdata.commandLine", "contains", "vbscript")]},
    ],
    "T1218.010": [
        {"name": "Regsvr32 suspicious usage",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "regsvr32.exe")]},
        {"name": "Regsvr32 loading remote SCT",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "regsvr32.exe"),
                        ("win.eventdata.commandLine", "contains", "http")]},
    ],
    "T1218.011": [
        {"name": "Rundll32 suspicious usage",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "rundll32.exe")]},
        {"name": "Rundll32 from AppData",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "rundll32.exe"),
                        ("win.eventdata.commandLine", "contains", "\\AppData\\")]},
    ],
    "T1105": [
        {"name": "Remote file download via PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "DownloadString")]},
        {"name": "Remote file download via certutil",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "certutil.exe"),
                        ("win.eventdata.commandLine", "contains", "urlcache")]},
        {"name": "Remote file download via bitsadmin",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "bitsadmin.exe"),
                        ("win.eventdata.commandLine", "contains", "/transfer")]},
        {"name": "Remote file download via curl",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "curl.exe"),
                        ("win.eventdata.commandLine", "contains", "http")]},
    ],

    # =========================================
    # PERSISTENCE
    # =========================================
    "T1547.001": [
        {"name": "Registry Run key write",
         "logsource": ("windows", "registry_event"),
         "conditions": [("win.eventdata.targetObject", "contains", "CurrentVersion\\Run")]},
        {"name": "Registry RunOnce key write",
         "logsource": ("windows", "registry_event"),
         "conditions": [("win.eventdata.targetObject", "contains", "CurrentVersion\\RunOnce")]},
    ],
    "T1547.004": [
        {"name": "Winlogon registry modification",
         "logsource": ("windows", "registry_event"),
         "conditions": [("win.eventdata.targetObject", "contains", "Winlogon"),
                        ("win.eventdata.targetObject", "contains", "Userinit")]},
    ],
    "T1543.003": [
        {"name": "New Windows service registered",
         "logsource": ("windows", "registry_event"),
         "conditions": [("win.eventdata.targetObject", "contains", "CurrentControlSet\\Services")]},
    ],
    "T1546.012": [
        {"name": "IFEO Debugger hijack",
         "logsource": ("windows", "registry_event"),
         "conditions": [("win.eventdata.targetObject", "contains", "Image File Execution Options"),
                        ("win.eventdata.details", "contains", "Debugger")]},
    ],
    "T1574.001": [
        {"name": "DLL search order hijack - AppData write",
         "logsource": ("windows", "file_event"),
         "conditions": [("win.eventdata.targetFilename", "contains", "\\AppData\\"),
                        ("win.eventdata.targetFilename", "endswith", ".dll")]},
    ],
    "T1574.002": [
        {"name": "DLL side-loading suspicious path",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "contains", "\\AppData\\"),
                        ("win.eventdata.commandLine", "contains", ".dll")]},
    ],
    "T1136.001": [
        {"name": "Local account creation via net",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "net.exe"),
                        ("win.eventdata.commandLine", "contains", "user /add")]},
    ],

    # =========================================
    # PRIVILEGE ESCALATION
    # =========================================
    "T1055": [
        {"name": "Process injection - PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "VirtualAlloc")]},
    ],
    "T1055.001": [
        {"name": "DLL injection via rundll32",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "rundll32.exe"),
                        ("win.eventdata.commandLine", "contains", "\\Temp\\")]},
    ],
    "T1055.012": [
        {"name": "Process hollowing indicators",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "NtUnmapViewOfSection")]},
    ],
    "T1068": [
        {"name": "Exploit privilege escalation via service",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.parentImage", "endswith", "services.exe"),
                        ("win.eventdata.image", "endswith", "cmd.exe")]},
    ],
    "T1134.001": [
        {"name": "Token impersonation via PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "ImpersonateLoggedOnUser")]},
    ],
    "T1134.002": [
        {"name": "Create process with token",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "CreateProcessWithToken")]},
    ],

    # =========================================
    # DEFENSE EVASION
    # =========================================
    "T1027": [
        {"name": "Encoded command indicators",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "-enc")]},
        {"name": "Base64 in command line",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "base64")]},
        {"name": "Reversed string execution",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "-join")]},
    ],
    "T1036": [
        {"name": "Masquerading - svchost from wrong parent",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "contains", "svchost"),
                        ("win.eventdata.parentImage", "endswith", "cmd.exe")]},
    ],
    "T1140": [
        {"name": "Deobfuscation via certutil decode",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "certutil.exe"),
                        ("win.eventdata.commandLine", "contains", "-decode")]},
    ],
    "T1562.001": [
        {"name": "Disable Windows Defender via PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "Set-MpPreference")]},
        {"name": "Disable firewall via netsh",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "netsh.exe"),
                        ("win.eventdata.commandLine", "contains", "off")]},
    ],
    "T1070.001": [
        {"name": "Event log cleared via wevtutil",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "wevtutil.exe"),
                        ("win.eventdata.commandLine", "contains", "cl")]},
    ],
    "T1070.004": [
        {"name": "File deletion via cmd del",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "cmd.exe"),
                        ("win.eventdata.commandLine", "contains", " del ")]},
        {"name": "File deletion via PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "Remove-Item")]},
    ],
    "T1497.001": [
        {"name": "Virtualization check via WMI",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "wmic.exe"),
                        ("win.eventdata.commandLine", "contains", "computersystem")]},
    ],

    # =========================================
    # CREDENTIAL ACCESS
    # =========================================
    "T1003.001": [
        {"name": "LSASS access via command",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "lsass")]},
        {"name": "Mimikatz sekurlsa",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "sekurlsa")]},
        {"name": "Mimikatz reference",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "mimikatz")]},
    ],
    "T1003.002": [
        {"name": "SAM hive save via reg.exe",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "reg.exe"),
                        ("win.eventdata.commandLine", "contains", "save")]},
    ],
    "T1003.003": [
        {"name": "NTDS.dit access via ntdsutil",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "ntdsutil.exe")]},
    ],
    "T1110.001": [
        {"name": "Password brute force via net",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "net.exe"),
                        ("win.eventdata.commandLine", "contains", "use")]},
    ],
    "T1110.003": [
        {"name": "Password spraying via PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "Invoke-Spray")]},
    ],
    "T1555.003": [
        {"name": "Browser credential theft",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "Login Data")]},
    ],
    "T1056.001": [
        {"name": "Keylogging via PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "GetAsyncKeyState")]},
    ],
    "T1539": [
        {"name": "Cookie theft via browser process",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "Cookies")]},
    ],

    # =========================================
    # DISCOVERY
    # =========================================
    "T1082": [
        {"name": "System info enumeration",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "systeminfo.exe")]},
    ],
    "T1046": [
        {"name": "Network scan via PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "Test-NetConnection")]},
        {"name": "Network scan via nmap",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "nmap")]},
    ],
    "T1087.001": [
        {"name": "Local account enumeration",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "net.exe"),
                        ("win.eventdata.commandLine", "contains", "user")]},
    ],
    "T1087.002": [
        {"name": "Domain account enumeration",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "net.exe"),
                        ("win.eventdata.commandLine", "contains", "domain")]},
    ],
    "T1083": [
        {"name": "File enumeration via dir",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "cmd.exe"),
                        ("win.eventdata.commandLine", "contains", "dir /s")]},
    ],
    "T1057": [
        {"name": "Process discovery via tasklist",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "tasklist.exe")]},
        {"name": "Process discovery via PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "Get-Process")]},
    ],
    "T1135": [
        {"name": "Network share enumeration",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "net.exe"),
                        ("win.eventdata.commandLine", "contains", "view")]},
    ],
    "T1069.001": [
        {"name": "Local group enumeration",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "net.exe"),
                        ("win.eventdata.commandLine", "contains", "localgroup")]},
    ],
    "T1069.002": [
        {"name": "Domain group enumeration",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "net.exe"),
                        ("win.eventdata.commandLine", "contains", "group /domain")]},
    ],
    "T1016": [
        {"name": "Network config discovery via ipconfig",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "ipconfig.exe")]},
        {"name": "Network config via PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "Get-NetIPAddress")]},
    ],
    "T1033": [
        {"name": "Current user discovery via whoami",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "whoami.exe")]},
    ],
    "T1018": [
        {"name": "Remote system discovery via ping",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "ping.exe"),
                        ("win.eventdata.commandLine", "contains", "-n")]},
        {"name": "Remote system discovery via nslookup",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "nslookup.exe")]},
    ],

    # =========================================
    # LATERAL MOVEMENT
    # =========================================
    "T1021.001": [
        {"name": "Outbound RDP connection",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "3389")]},
    ],
    "T1021.002": [
        {"name": "SMB lateral movement",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "445")]},
    ],
    "T1021.006": [
        {"name": "WinRM lateral movement",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "5985")]},
    ],
    "T1550.002": [
        {"name": "Pass the hash via wmic",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "wmic.exe"),
                        ("win.eventdata.commandLine", "contains", "/node:")]},
    ],
    "T1570": [
        {"name": "Lateral tool transfer via SMB",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "445"),
                        ("win.eventdata.image", "endswith", "cmd.exe")]},
    ],
    "T1534": [
        {"name": "Internal spearphishing via Outlook",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.parentImage", "contains", "OUTLOOK"),
                        ("win.eventdata.image", "endswith", "powershell.exe")]},
    ],
    "T1080": [
        {"name": "Taint shared content - write to share",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "\\\\"),
                        ("win.eventdata.commandLine", "contains", "copy")]},
    ],

    # =========================================
    # COLLECTION
    # =========================================
    "T1560.001": [
        {"name": "Archive via 7zip or WinRAR",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "contains", "7z"),
                        ("win.eventdata.commandLine", "contains", " a ")]},
        {"name": "Archive via PowerShell compress",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "Compress-Archive")]},
    ],
    "T1005": [
        {"name": "Local data collection via xcopy",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "xcopy.exe")]},
        {"name": "Local data collection via robocopy",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "robocopy.exe")]},
    ],
    "T1074.001": [
        {"name": "Local data staged in temp folder",
         "logsource": ("windows", "file_event"),
         "conditions": [("win.eventdata.targetFilename", "contains", "\\Temp\\")]},
    ],
    "T1113": [
        {"name": "Screenshot via PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "CopyFromScreen")]},
    ],
    "T1115": [
        {"name": "Clipboard access via PowerShell",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "powershell.exe"),
                        ("win.eventdata.commandLine", "contains", "Get-Clipboard")]},
    ],

    # =========================================
    # COMMAND AND CONTROL
    # =========================================
    "T1071.001": [
        {"name": "Outbound HTTPS from PowerShell",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "443"),
                        ("win.eventdata.image", "endswith", "powershell.exe")]},
        {"name": "Outbound HTTP from PowerShell",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "80"),
                        ("win.eventdata.image", "endswith", "powershell.exe")]},
    ],
    "T1071.004": [
        {"name": "Suspicious DNS query",
         "logsource": ("windows", "dns_query"),
         "conditions": [("win.eventdata.queryName", "contains", ".onion")]},
    ],
    "T1090": [
        {"name": "Proxy connection via netsh",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "netsh.exe"),
                        ("win.eventdata.commandLine", "contains", "portproxy")]},
    ],
    "T1090.003": [
        {"name": "Tor exit node network connection",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "9050")]},
    ],
    "T1095": [
        {"name": "Non-standard protocol raw socket",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "4444")]},
    ],
    "T1571": [
        {"name": "Non-standard port 8080",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "8080")]},
        {"name": "Non-standard port 8443",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "8443")]},
    ],
    "T1573": [
        {"name": "Encrypted channel via HTTPS",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "443"),
                        ("win.eventdata.image", "endswith", "powershell.exe")]},
    ],
    "T1573.001": [
        {"name": "Symmetric encryption C2",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "AES")]},
    ],
    "T1573.002": [
        {"name": "Asymmetric encryption C2",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "RSA")]},
    ],
    "T1132": [
        {"name": "Data encoding via certutil",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "certutil.exe"),
                        ("win.eventdata.commandLine", "contains", "-encode")]},
    ],
    "T1001": [
        {"name": "Data obfuscation in traffic",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "base64"),
                        ("win.eventdata.image", "endswith", "powershell.exe")]},
    ],
    "T1008": [
        {"name": "Fallback C2 channel",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "80"),
                        ("win.eventdata.image", "endswith", "powershell.exe")]},
    ],

    # =========================================
    # EXFILTRATION
    # =========================================
    "T1041": [
        {"name": "Exfil over C2 HTTPS",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "443"),
                        ("win.eventdata.image", "endswith", "powershell.exe")]},
    ],
    "T1567": [
        {"name": "Exfil to web service via PowerShell",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "443"),
                        ("win.eventdata.image", "endswith", "powershell.exe")]},
    ],
    "T1567.002": [
        {"name": "Cloud storage exfiltration via PowerShell",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "443"),
                        ("win.eventdata.image", "endswith", "powershell.exe")]},
        {"name": "Cloud storage upload via cmd",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "upload"),
                        ("win.eventdata.image", "endswith", "cmd.exe")]},
    ],
    "T1048": [
        {"name": "Exfil over alternative protocol",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "21")]},
    ],
    "T1048.003": [
        {"name": "Exfil over unencrypted protocol",
         "logsource": ("windows", "network_connection"),
         "conditions": [("win.eventdata.destinationPort", "equals", "21"),
                        ("win.eventdata.image", "endswith", "ftp.exe")]},
    ],

    # =========================================
    # IMPACT
    # =========================================
    "T1490": [
        {"name": "Shadow copy deletion via vssadmin",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "vssadmin.exe"),
                        ("win.eventdata.commandLine", "contains", "delete")]},
        {"name": "BCDedit recovery disabled",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "bcdedit.exe"),
                        ("win.eventdata.commandLine", "contains", "recoveryenabled")]},
    ],
    "T1486": [
        {"name": "Ransomware file extension indicator",
         "logsource": ("windows", "file_event"),
         "conditions": [("win.eventdata.targetFilename", "contains", ".encrypted")]},
        {"name": "Mass file modification pattern",
         "logsource": ("windows", "file_event"),
         "conditions": [("win.eventdata.targetFilename", "contains", ".locked")]},
    ],
    "T1489": [
        {"name": "Service stop via net stop",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "net.exe"),
                        ("win.eventdata.commandLine", "contains", "stop")]},
        {"name": "Service stop via sc.exe",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.image", "endswith", "sc.exe"),
                        ("win.eventdata.commandLine", "contains", "stop")]},
    ],
    "T1491.001": [
        {"name": "Local defacement - wallpaper change",
         "logsource": ("windows", "process_creation"),
         "conditions": [("win.eventdata.commandLine", "contains", "Wallpaper")]},
    ],
}

In [ ]:
SIGMA_TO_WAZUH_FIELD_MAP = {
    "Image":           "win.eventdata.image",
    "ParentImage":     "win.eventdata.parentImage",
    "CommandLine":     "win.eventdata.commandLine",
    "DestinationIp":   "win.eventdata.destinationIp",
    "SourceIp":        "win.eventdata.sourceIp",
    "DestinationPort": "win.eventdata.destinationPort",
    "QueryName":       "win.eventdata.queryName",
    "TargetObject":    "win.eventdata.targetObject",
    "Details":         "win.eventdata.details",
    "Hashes":          "win.eventdata.hashes",
}

SIGMA_RELEVANT_FIELDS_BY_IOC = {
    "ip": {
        "win.eventdata.destinationIp",
        "win.eventdata.destinationPort",
        "win.eventdata.image",
    },
    "domain": {
        "win.eventdata.queryName",
        "win.eventdata.destinationIp",
        "win.eventdata.image",
    },
    "url": {
        "win.eventdata.url",
        "win.eventdata.destinationIp",
        "win.eventdata.image",
    },
    "file_hash": {
        "win.eventdata.hashes",
        "win.eventdata.image",
        "win.eventdata.commandLine",
    },
    "process_command": {
        "win.eventdata.commandLine",
        "win.eventdata.image",
        "win.eventdata.parentImage",
    },
    "registry_key": {
        "win.eventdata.targetObject",
        "win.eventdata.details",
    },
    "email": {
        "win.eventdata.image",
        "win.eventdata.commandLine",
    },
}

import re as _re

def _extract_context_keywords(context: str) -> set:
    stop_words = {
        "the", "a", "an", "is", "in", "on", "at", "to", "for",
        "of", "and", "or", "by", "from", "with", "this", "that",
        "was", "are", "has", "have", "been", "its", "it", "as",
        "via", "into", "over", "through", "using", "used", "known",
        "observed", "internal", "external", "destination", "source",
    }
    tokens = _re.findall(r"[a-zA-Z0-9_\-\.]{3,}", context.lower())
    return {t for t in tokens if t not in stop_words}


def _score_sigma_rule_relevance(rule_conditions, ioc_type, context_keywords, ioc_value) -> int:
    relevant_fields = SIGMA_RELEVANT_FIELDS_BY_IOC.get(ioc_type, set())
    ioc_lower       = ioc_value.lower()
    score           = 0

    for field, op, value in rule_conditions:
        value_lower = value.lower()
        if field in relevant_fields:
            score += 3
        else:
            score -= 2
        if any(kw in value_lower for kw in context_keywords):
            score += 2
        if ioc_lower and ioc_lower in value_lower:
            score += 5

    return score


def mine_sigma_constraints(
    sigma_rules,
    ioc_type,
    ioc_value: str = "",
    context: str = "",
    max_constraints_per_rule: int = 3,
    relevance_threshold: int = 4,
):
    anchor          = IOC_ANCHOR_FIELD.get(ioc_type)
    ctx_keywords    = _extract_context_keywords(context) if context else set()

    scored_rules = []

    for r in sigma_rules:
        detection  = r.get("detection", {}) or {}
        conditions = sigma_detection_to_wazuh_conditions(detection)

        safe = []
        for field, modifier, value in conditions:
            if field not in ALLOWED_WAZUH_FIELDS:
                continue
            if anchor and field == anchor:
                continue
            op = modifier if modifier in ALLOWED_OPS else "equals"
            v  = str(value)
            if len(v) > 120:
                continue
            safe.append((field, op, v))
            if len(safe) >= max_constraints_per_rule:
                break

        if not safe:
            continue

        score = _score_sigma_rule_relevance(
            rule_conditions=safe,
            ioc_type=ioc_type,
            context_keywords=ctx_keywords,
            ioc_value=ioc_value,
        )

        if score < relevance_threshold:
            print(f"  ⚠️  Sigma rule skipped (score={score}): {r.get('title', 'unknown')}")
            continue

        scored_rules.append((score, {
            "sigma_title":     r.get("title"),
            "sigma_id":        r.get("id"),
            "conditions":      safe,
            "relevance_score": score,
        }))

    scored_rules.sort(key=lambda x: x[0], reverse=True)
    mined = [rule for _, rule in scored_rules]

    print(f"✅ Sigma mining: {len(mined)} relevant rules kept, "
          f"{len(sigma_rules) - len(mined)} filtered out")

    return mined

In [ ]:
def resolve_parent_sid_from_logsource(logsource_tuple):
    return LOGSOURCE_TO_PARENT_SID.get(logsource_tuple)


def sanitize_conditions(conditions, ioc_value):
    """
    Replace $IOC, enforce allowlists, enforce ops.
    """
    out = []
    for field, op, value in conditions:
        if field not in ALLOWED_WAZUH_FIELDS:
            continue
        if op not in ALLOWED_OPS:
            op = "equals"
        v = str(value).replace("$IOC", str(ioc_value))
        out.append((field, op, v))
    return out


def risk_score_to_wazuh_level(risk_score: int) -> int:
    """
    Map enrichment risk score (0-100) to Wazuh alert level (1-15).

    Wazuh level semantics:
      1-3   = informational / noise
      4-6   = low priority
      7-9   = medium — generates alerts in most configs
      10-12 = high — triggers active response in many setups
      13-14 = critical
      15    = maximum — reserved for confirmed attacks

    We use:
      0        → 5  (unknown/unenriched — informational)
      1-39     → 7  (low)
      40-64    → 10 (medium)
      65-84    → 13 (high)
      85-100   → 15 (critical)
    """
    if risk_score >= 85:
        return 15
    if risk_score >= 65:
        return 13
    if risk_score >= 40:
        return 10
    if risk_score > 0:
        return 7
    return 5   # unknown / no enrichment data


def build_wazuh_rule_xml(
    rule_id,
    parent_sid,
    description,
    mitre_ids,
    conditions,
    level: int = 10           # ← NEW param (default 10 = medium, safe fallback)
):
    if not parent_sid or not conditions:
        return None

    # Clamp level to valid Wazuh range
    level = max(1, min(15, int(level)))

    mitre_xml = build_mitre_block([m.upper() for m in mitre_ids if m])

    field_blocks = []
    for field, op, value in conditions:
        field_xml = build_field_xml(field, op, value)
        if field_xml:
            field_blocks.append(field_xml)

    if not field_blocks:
        return None

    fields_xml = "\n    ".join(field_blocks)

    return f"""
<group name="windows,attack">
  <rule id="{rule_id}" level="{level}">
    <if_sid>{parent_sid}</if_sid>
    {fields_xml}
    <description>{description}</description>
    {mitre_xml}
  </rule>
</group>
""".strip()

In [ ]:
# =========================================
# ✅ HYBRID CANDIDATE GENERATION
# Produces a "menu" for the SOC analyst
# =========================================

import hashlib

def _stable_rule_id(ioc_type: str, ioc_value: str, suffix: str) -> int:
    """
    Stable Wazuh rule ID derived from IOC identity + rule type.
    Same IOC always gets the same rule IDs — no conflicts on re-runs.
    Range: 100000–109999 (safe custom rule range in Wazuh).
    """
    raw = f"{ioc_type}:{ioc_value}:{suffix}"
    h   = int(hashlib.sha256(raw.encode()).hexdigest(), 16)
    return 100000 + (h % 10000)


def generate_candidate_rules(
    ioc_type,
    ioc_value,
    validated_techniques,
    sigma_rules_for_techniques,
    limit_per_bucket=5,
    risk_score: int = 0,
    context: str = "",
):
    candidates = []
    seen_ids   = set()   # prevent hash collisions within one run

    def next_id(suffix):
        rid = _stable_rule_id(ioc_type, ioc_value, suffix)
        # On the rare collision, increment until free
        while rid in seen_ids:
            rid += 1
        seen_ids.add(rid)
        return rid

    base_level      = risk_score_to_wazuh_level(risk_score)
    blueprint_level = max(5, base_level - 1)
    sigma_level     = max(5, base_level - 2)

    # -------------------------
    # A) IOC-EXACT BASE RULE
    # -------------------------
    base_bp = IOC_BASE_BLUEPRINT.get(ioc_type)
    if base_bp:
        parent_sid      = resolve_parent_sid_from_logsource(base_bp["logsource"])
        base_conditions = sanitize_conditions(base_bp["conditions"], ioc_value)
        rule_id         = next_id("ioc_exact")

        xml = build_wazuh_rule_xml(
            rule_id=rule_id,
            parent_sid=parent_sid,
            description=f"[IOC-EXACT] {ioc_type} match: {ioc_value}",
            mitre_ids=validated_techniques,
            conditions=base_conditions,
            level=base_level
        )
        if xml:
            candidates.append({
                "candidate_type": "ioc_exact",
                "rule_id":        rule_id,
                "description":    f"Base IOC match for {ioc_type}",
                "mitre":          validated_techniques,
                "wazuh_xml":      xml,
                "wazuh_level":    base_level,
            })

    # -------------------------
    # B) TECHNIQUE BLUEPRINTS
    # -------------------------
    for tid in validated_techniques:
        blueprints = TECHNIQUE_BLUEPRINTS.get(tid, [])
        for i, bp in enumerate(blueprints[:limit_per_bucket]):
            parent_sid = resolve_parent_sid_from_logsource(bp["logsource"])
            conds      = sanitize_conditions(bp["conditions"], ioc_value)
            rule_id    = next_id(f"{tid}_{i}")

            xml = build_wazuh_rule_xml(
                rule_id=rule_id,
                parent_sid=parent_sid,
                description=f"[BLUEPRINT] {tid} - {bp['name']}",
                mitre_ids=[tid],
                conditions=conds,
                level=blueprint_level
            )
            if xml:
                candidates.append({
                    "candidate_type": "blueprint",
                    "rule_id":        rule_id,
                    "description":    bp["name"],
                    "mitre":          [tid],
                    "wazuh_xml":      xml,
                    "wazuh_level":    blueprint_level,
                })

    # -------------------------
    # C) SIGMA-ENRICHED VARIANTS
    # -------------------------
    mined = mine_sigma_constraints(
        sigma_rules_for_techniques,
        ioc_type=ioc_type,
        ioc_value=ioc_value,
        context=context,
        max_constraints_per_rule=3
    )

    for m in mined[:limit_per_bucket]:
        if not base_bp:
            continue

        parent_sid      = resolve_parent_sid_from_logsource(base_bp["logsource"])
        base_conditions = sanitize_conditions(base_bp["conditions"], ioc_value)
        extra_conditions= sanitize_conditions(m["conditions"], ioc_value)
        combined        = base_conditions + extra_conditions
        rule_id         = next_id(f"sigma_{m.get('sigma_id', m.get('sigma_title', 'unknown'))}")

        xml = build_wazuh_rule_xml(
            rule_id=rule_id,
            parent_sid=parent_sid,
            description=f"[SIGMA-ENRICHED] + {m.get('sigma_title', 'sigma')}",
            mitre_ids=validated_techniques,
            conditions=combined,
            level=sigma_level
        )
        if xml:
            candidates.append({
                "candidate_type": "sigma_enriched",
                "rule_id":        rule_id,
                "description":    m.get("sigma_title"),
                "sigma_id":       m.get("sigma_id"),
                "mitre":          validated_techniques,
                "wazuh_xml":      xml,
                "wazuh_level":    sigma_level,
            })

    return candidates

In [ ]:
# =========================================
# ✅ DEDUP + IOC ANCHOR FIELD PROTECTION
# =========================================

IOC_ANCHOR_FIELD = {
    "ip": "win.eventdata.destinationIp",
    "domain": "win.eventdata.queryName",
    "url": "win.eventdata.url",
    "file_hash": "win.eventdata.hashes",
    "email": "win.eventdata.sender",          # adjust if your email telemetry differs
    "process_command": "win.eventdata.commandLine",
    "registry_key": "win.eventdata.targetObject",
}

def rule_signature(parent_sid, conditions):
    """
    Create stable signature for dedup.
    Normalize to tuples and sort conditions.
    """
    norm = []
    for f, op, v in conditions:
        norm.append((str(f).lower(), str(op).lower(), str(v).lower()))
    norm.sort()
    return (int(parent_sid), tuple(norm))

def remove_duplicates(candidates):
    """
    Dedup by (parent_sid + normalized condition set).
    Keeps first occurrence.
    """
    seen = set()
    out = []
    for c in candidates:
        sig = c.get("_sig")
        if not sig:
            out.append(c)
            continue
        if sig in seen:
            continue
        seen.add(sig)
        c.pop("_sig", None)  # remove internal key
        out.append(c)
    return out

In [ ]:
def full_pipeline(ioc_type: str, ioc_value: str, context: str, base_dir: str):

    # ================================
    # 1️⃣ LLM → MITRE Mapping
    # ================================
    llm_output = llm_only_map(
        ioc_type=ioc_type,
        ioc=ioc_value,
        context=context
    )

    # ================================
    # 2️⃣ RAG Validation
    # ================================
    validated = rag_validate_techniques(
        llm_output=llm_output,
        attack_index=ATTACK_ID_SET,
        evidence_profiles=evidence_profiles["ioc_type_to_evidence"]
    )

    # ================================
    # 3️⃣ Enrich Validated Techniques
    # ================================
    enriched_techniques = []
    for t in validated.get("techniques", []):
        tid        = t["id"]
        mitre_meta = ATTACK_META_INDEX.get(tid, {})
        enriched_techniques.append({
            "id":           tid,
            "name":         mitre_meta.get("name"),
            "tactics":      mitre_meta.get("tactics", []),
            "description":  mitre_meta.get("description"),
            "reason":       t.get("reason"),
            "validated_by": t.get("validated_by", "RAG_ATT&CK")
        })

    validated_technique_ids = [t["id"] for t in enriched_techniques]
    if not validated_technique_ids:
        return {
            "status": "no_validated_techniques",
            "reason": "LLM output did not pass RAG validation",
            "rejected": validated.get("rejected", [])
        }

    # ================================
    # 4️⃣ Load & Normalize Sigma Rules
    # ================================
    sigma_rules      = load_sigma_rules(base_dir)
    normalized_rules = [normalize_sigma_rule(r) for r in sigma_rules]

    # ================================
    # 5️⃣ Filter Sigma by MITRE + IOC type
    # ================================
    sigma_by_mitre = filter_sigma_by_mitre(normalized_rules, validated_technique_ids)
    sigma_by_ioc   = filter_by_ioc_type(sigma_by_mitre, ioc_type)

    # ================================
    # 6️⃣ Get risk score from enrichment
    # ================================
    risk_score = full_pipeline._risk_score if hasattr(full_pipeline, "_risk_score") else 0

    # ================================
    # 7️⃣ Generate Hybrid Candidate Bundle
    # ================================
    candidates = generate_candidate_rules(
        ioc_type=ioc_type,
        ioc_value=ioc_value,
        validated_techniques=validated_technique_ids,
        sigma_rules_for_techniques=sigma_by_ioc,
        limit_per_bucket=5,
        risk_score=risk_score,
        context=context
    )

    return {
        "status":               "success",
        "techniques":           enriched_techniques,
        "validated_techniques": validated_technique_ids,
        "rejected_techniques":  validated.get("rejected", []),
        "candidate_rules":      candidates,
        "risk_score_used":      risk_score,
        "note": "Hybrid output: IOC-exact + technique blueprints + sigma-enriched variants. "
                "Rule levels computed from enrichment risk score."
    }

In [ ]:
!curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
!echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list >/dev/null
!sudo apt update -qq && sudo apt install ngrok -y -qq
!ngrok config add-authtoken 3922FPsN6o4KTHUt0hMRN2qQ2yp_5uJcFz2spFU5vB6Nv7dB8

^C
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok


In [ ]:
from fastapi import FastAPI
import nest_asyncio
import json, torch, subprocess, threading, time, requests

nest_asyncio.apply()

app = FastAPI(title="ATT&CKsmith Pipeline API")

SIGMA_BASE_DIR = "sigma/rules"

@app.post("/run")
async def run_pipeline(request: dict):
    if request.get("mode") is None:
        full_pipeline._risk_score = int(request.get("risk_score", 0) or 0)
        return full_pipeline(
            ioc_type=request.get("ioc_type"),
            ioc_value=request.get("ioc_value"),
            context=request.get("context"),
            base_dir=SIGMA_BASE_DIR
        )

    if request.get("mode") == "next_technique_prediction":
        payload = request.get("payload", {})
        prompt = build_apt_projection_prompt(
            ioc_type="projection", ioc="N/A",
            context=payload.get("context", ""),
            detected_techniques=payload.get("current_techniques", []) or [],
            candidate_apts=payload.get("candidate_apts", []) or [],
            candidate_next_techniques=(payload.get("candidate_next_techniques", []) or [])[:50],
            enrichment_summary=payload.get("enrichment_summary", None)
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=256,
                                    do_sample=False, pad_token_id=tokenizer.eos_token_id)
        raw = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        try:
            return json.loads(raw[raw.find("{"):raw.rfind("}")+1])
        except:
            return {"error": "parsing failed", "raw_output": raw}

    return {"error": "Unsupported mode"}

# ── Start uvicorn using uvicorn.Config to avoid nest_asyncio conflict ──
import uvicorn, asyncio

async def start_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    await server.serve()

loop = asyncio.get_event_loop()
loop.create_task(start_server())
time.sleep(2)

# ── Start ngrok ──
ngrok_proc = subprocess.Popen(
    ["ngrok", "http", "8000",
     "--authtoken", "3922FPsN6o4KTHUt0hMRN2qQ2yp_5uJcFz2spFU5vB6Nv7dB8"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)

try:
    resp = requests.get("http://localhost:4040/api/tunnels")
    public_url = resp.json()["tunnels"][0]["public_url"]
    print(f"✅ API live at: {public_url}")
except:
    print("❌ Could not get ngrok URL — check ngrok is installed")

✅ API live at: https://unerased-oxymoronically-tabitha.ngrok-free.dev
